# BERT WSI

Word Sense Induction baseline using RuBERT contextual embeddings for the target word.

In [ ]:
import re
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import adjusted_rand_score
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

In [ ]:
df = pd.read_csv("data/train/train.csv", sep="\t")
df = df.drop(columns=["predict_sense_id"], errors="ignore")

print(f"Rows: {len(df)}")
print(f"Words: {df['word'].nunique()}")
print(f"Missing positions: {df['positions'].isna().sum()}")
df.head()

In [ ]:
def build_validation_split(data, valid_frac=0.2, random_state=42):
    rng = np.random.RandomState(random_state)
    train_idx = []
    valid_idx = []

    for _, word_df in data.groupby("word"):
        for _, sense_df in word_df.groupby("gold_sense_id"):
            indices = sense_df.index.to_numpy().copy()
            rng.shuffle(indices)

            if len(indices) < 2:
                train_idx.extend(indices.tolist())
                continue

            n_valid = max(1, int(round(len(indices) * valid_frac)))
            n_valid = min(n_valid, len(indices) - 1)

            valid_idx.extend(indices[:n_valid].tolist())
            train_idx.extend(indices[n_valid:].tolist())

    return sorted(train_idx), sorted(valid_idx)


def score_wordwise(valid_df, predictions):
    scored_df = valid_df[["word", "gold_sense_id"]].copy()
    scored_df["prediction"] = pd.Series(predictions, index=valid_df.index).astype(str)
    scored_df["gold_sense_id"] = scored_df["gold_sense_id"].astype(str)

    word_scores = []
    for word, word_df in scored_df.groupby("word"):
        ari = adjusted_rand_score(word_df["gold_sense_id"], word_df["prediction"])
        word_scores.append((word, ari, len(word_df)))

    scores_df = pd.DataFrame(word_scores, columns=["word", "ari", "n_valid"])
    macro_ari = scores_df["ari"].mean()
    weighted_ari = np.average(scores_df["ari"], weights=scores_df["n_valid"])
    return macro_ari, weighted_ari, scores_df


def summarize_runs(runs_df):
    return (
        runs_df.groupby("model", as_index=False)
        .agg(
            macro_ari_mean=("macro_ari", "mean"),
            macro_ari_std=("macro_ari", "std"),
            weighted_ari_mean=("weighted_ari", "mean"),
            weighted_ari_std=("weighted_ari", "std"),
        )
        .sort_values("macro_ari_mean", ascending=False)
    )

In [ ]:
def common_prefix_len(left, right):
    prefix_len = 0
    for left_char, right_char in zip(left, right):
        if left_char != right_char:
            break
        prefix_len += 1
    return prefix_len


def parse_target_span(positions, text, word):
    positions_str = "" if pd.isna(positions) else str(positions)

    for chunk in positions_str.split(","):
        chunk = chunk.strip()
        if "-" not in chunk:
            continue
        start, end = chunk.split("-", 1)
        return int(start), int(end)

    text_lower = str(text).lower()
    word_lower = str(word).lower()
    exact_matches = [match.span() for match in re.finditer(re.escape(word_lower), text_lower)]
    if len(exact_matches) == 1:
        return exact_matches[0]

    token_matches = []
    for match in re.finditer(r"[A-Za-zА-Яа-яЁё-]+", str(text)):
        token = match.group(0)
        prefix_len = common_prefix_len(token.lower(), word_lower)
        min_prefix = max(3, min(len(token), len(word_lower)) - 2)
        if prefix_len >= min_prefix:
            token_matches.append(match.span())

    if len(token_matches) == 1:
        return token_matches[0]

    raise ValueError(f"Could not infer target span for word={word!r}, positions={positions!r}")


def resolve_positions(row):
    try:
        start, end = parse_target_span(row.positions, row.context, row.word)
        return f"{start}-{end}"
    except ValueError:
        return np.nan


def prepare_span_dataset(data):
    span_df = data.copy()
    span_df["resolved_positions"] = span_df.apply(resolve_positions, axis=1)
    failed_rows = span_df[span_df["resolved_positions"].isna()][["context_id", "word", "context"]].copy()
    span_df = span_df[span_df["resolved_positions"].notna()].copy()
    span_df["positions"] = span_df["resolved_positions"]
    span_df = span_df.drop(columns=["resolved_positions"])
    return span_df, failed_rows

In [ ]:
MODEL_NAME = "cointegrated/rubert-tiny2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
bert_model.eval()

DEVICE

In [ ]:
def extract_target_embedding(text, positions, word, tokenizer, model, device):
    start_char, end_char = parse_target_span(positions, text, word)
    encoded = tokenizer(
        text,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True,
        max_length=512,
    )

    offset_mapping = encoded.pop("offset_mapping")[0].tolist()
    model_inputs = {key: value.to(device) for key, value in encoded.items()}

    with torch.no_grad():
        outputs = model(**model_inputs)

    hidden = outputs.last_hidden_state[0]
    token_indices = []

    for idx, (token_start, token_end) in enumerate(offset_mapping):
        if token_end <= token_start:
            continue
        if token_end <= start_char:
            continue
        if token_start >= end_char:
            continue
        token_indices.append(idx)

    if not token_indices:
        token_indices = [1]

    embedding = hidden[token_indices].mean(dim=0)
    return embedding.detach().cpu().numpy()


def build_bert_matrix(data, tokenizer, model, device):
    vectors = []
    for row in tqdm(data.itertuples(index=False), total=len(data)):
        vectors.append(
            extract_target_embedding(
                text=row.context,
                positions=row.positions,
                word=row.word,
                tokenizer=tokenizer,
                model=model,
                device=device,
            )
        )
    return np.vstack(vectors)

In [ ]:
def run_bert_validation(data, random_state=42, valid_frac=0.2):
    train_idx, valid_idx = build_validation_split(data, valid_frac=valid_frac, random_state=random_state)
    train_df = data.loc[train_idx].copy()
    valid_df = data.loc[valid_idx].copy()

    train_bert_df, train_failed = prepare_span_dataset(train_df)
    valid_bert_df, valid_failed = prepare_span_dataset(valid_df)

    x_train = build_bert_matrix(train_bert_df.reset_index(drop=True), tokenizer, bert_model, DEVICE)
    x_valid = build_bert_matrix(valid_bert_df.reset_index(drop=True), tokenizer, bert_model, DEVICE)

    predictions = pd.Series(index=valid_bert_df.index, dtype="object")
    for word in sorted(valid_bert_df["word"].unique()):
        train_mask = (train_bert_df["word"] == word).to_numpy()
        valid_mask = (valid_bert_df["word"] == word).to_numpy()
        y_train = train_bert_df.loc[train_mask, "gold_sense_id"].astype(str)

        if y_train.nunique() == 1:
            pred = np.repeat(y_train.iloc[0], valid_mask.sum())
        else:
            clf = LogisticRegression(max_iter=1000, random_state=random_state)
            clf.fit(x_train[train_mask], y_train)
            pred = clf.predict(x_valid[valid_mask])

        predictions.loc[valid_bert_df.index[valid_mask]] = pred

    macro_ari, weighted_ari, scores_df = score_wordwise(valid_bert_df, predictions)
    metadata = {"train_unresolved": len(train_failed), "valid_unresolved": len(valid_failed)}
    return macro_ari, weighted_ari, scores_df, metadata

In [ ]:
random_states = [13, 21, 34, 42, 55]
runs = []
word_scores = []

for random_state in random_states:
    macro_ari, weighted_ari, scores_df, metadata = run_bert_validation(df, random_state=random_state)
    runs.append((
        "bert_target_embedding",
        random_state,
        macro_ari,
        weighted_ari,
        metadata["train_unresolved"],
        metadata["valid_unresolved"],
    ))

    scores_df = scores_df.copy()
    scores_df["model"] = "bert_target_embedding"
    scores_df["random_state"] = random_state
    word_scores.append(scores_df)

bert_runs_df = pd.DataFrame(
    runs,
    columns=["model", "random_state", "macro_ari", "weighted_ari", "train_unresolved", "valid_unresolved"],
)
bert_word_scores_df = pd.concat(word_scores, ignore_index=True)
summarize_runs(bert_runs_df)